# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YomnaImad07/FlyRank-ML-Internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## 1. Two paper findings + my methodology questions

**Finding 1 — Position volatility predicts decline.**
Decline rate rose monotonically across position_volatility quartiles: 37.4% in the
most stable quartile up to 59.8% in the most volatile, a 17-point spread with no
reversals (Week 4 signal audit).

*Methodology question:* The label (is_declining) is defined from the same fact_daily
table that position_volatility is computed from, just a different time slice. Does the
90-day volatility window fully precede the 30-day outcome window used for the label,
or is there any date overlap between the two? Worth double-checking the exact interval
boundaries rather than assuming the split is clean just because the columns have
different names.

**Finding 2 — top_query_share flag identifies at-risk pages.**
Pages with top_query_share > 0.5 showed a decline rate of 48.6% (n=14,328), essentially
the same as the unflagged group's 50.1% (n=62,827) — no meaningful difference (Week 4
signal audit).

*Methodology question:* This test used a single fixed threshold (0.5) picked without
justification. Before concluding the flag itself is broken, does the result hold across
a few different thresholds (e.g. 0.3, 0.4, 0.6), or does the null result only appear at
0.5 specifically? A flag that's miscalibrated (wrong threshold) is a different problem
from a flag whose underlying signal is genuinely useless — the validation design here
doesn't distinguish between the two.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [8]:
%pip -q install duckdb huggingface_hub

import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':       f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':       f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':        f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}


span = con.sql(f"""
    SELECT MIN(report_date) AS min_d, MAX(report_date) AS max_d,
           DATEDIFF('day', MIN(report_date), MAX(report_date)) AS total_days
    FROM {TABLES['fact_daily']}
""").df()
half_window = span['total_days'].iloc[0] // 2
print("half_window:", half_window)

FINAL_THRESHOLD = 100

features = con.sql(f"""
    WITH bounds AS (SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL {half_window} DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL {half_window} DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL {half_window} DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last30,
               AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL {half_window} DAY THEN f.gsc_avg_position END)       AS pos_last30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL {half_window * 2} DAY
        GROUP BY 1, 2
        HAVING imp_prev30 >= {FINAL_THRESHOLD}
    )
    SELECT * FROM windowed
""").df()

qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count)     AS visible_queries,
           ANY_VALUE(rare_impressions_share)          AS rare_share,
           ANY_VALUE(anonymized_impressions_share)    AS anon_share,
           MAX(impressions_90d)                       AS top_query_impressions,
           SUM(impressions_90d)                       AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

print(f'features: {len(features):,} rows | qsignals: {len(qsignals):,} rows')

data = features.merge(qsignals, on='content_hash_id', how='left')

volatility = con.sql(f"""
    WITH bounds AS (SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']})
    SELECT f.content_hash_id, STDDEV(f.gsc_avg_position) AS position_volatility
    FROM {TABLES['fact_daily']} f, bounds b
    WHERE f.report_date > b.end_d - INTERVAL {half_window * 2} DAY
    GROUP BY 1
""").df()
data = data.merge(volatility, on='content_hash_id', how='left')

data['top_query_share'] = data['top_query_impressions'] / data['kept_impressions']

fill_cols = ['visible_queries', 'rare_share', 'anon_share', 'top_query_share', 'position_volatility']
data[fill_cols] = data[fill_cols].fillna(0)

data['is_declining'] = (data['imp_last30'] < 0.8 * data['imp_prev30']).astype(int)

feature_cols = ['imp_prev30', 'visible_queries', 'rare_share',
                 'anon_share', 'top_query_share', 'position_volatility']

print("data rows:", len(data))
data.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

half_window: 14


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

features: 77,155 rows | qsignals: 133,852 rows
data rows: 77155


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30,visible_queries,rare_share,anon_share,top_query_impressions,kept_impressions,position_volatility,top_query_share,is_declining
0,client_3ffa76342f366962,content_0e92904e80e0ebf2,129.0,188.0,2.0,7.043259,6.0,0.014749,0.839725,65.0,148.0,3.366924,0.439189,1
1,client_e547b89c05043229,content_a9589a5b2b6b41b8,10577.0,11146.0,23.0,7.390199,160.0,0.016274,0.709784,1168.0,15284.0,0.478296,0.076420,0
2,client_e547b89c05043229,content_0230669d6fc4a51d,208.0,211.0,0.0,7.682568,5.0,0.139506,0.806173,38.0,88.0,4.840435,0.431818,0
3,client_e547b89c05043229,content_d1b697c9c74ff65c,298.0,171.0,1.0,31.548775,12.0,0.253469,0.561517,52.0,200.0,12.201565,0.260000,0
4,client_e547b89c05043229,content_38e4844afae585d9,2495.0,1578.0,6.0,9.979458,45.0,0.055271,0.611435,774.0,2798.0,0.820766,0.276626,0


In [9]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score

model_data = data.dropna(subset=feature_cols).reset_index(drop=True)
X, y = model_data[feature_cols], model_data['is_declining']
groups = model_data['client_hash_id']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
clf_before = RandomForestClassifier(n_estimators=200, random_state=42)
clf_before.fit(X_tr, y_tr)
pred_before = clf_before.predict(X_te)
f1_before = f1_score(y_te, pred_before)

print("=== BEFORE: random split ===")
print(classification_report(y_te, pred_before, digits=3))

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
X_tr_g, X_te_g = X.iloc[train_idx], X.iloc[test_idx]
y_tr_g, y_te_g = y.iloc[train_idx], y.iloc[test_idx]

clf_after = RandomForestClassifier(n_estimators=200, random_state=42)
clf_after.fit(X_tr_g, y_tr_g)
pred_after = clf_after.predict(X_te_g)
f1_after = f1_score(y_te_g, pred_after)

print("\n=== AFTER: grouped-by-client split ===")
print(classification_report(y_te_g, pred_after, digits=3))

print(f"\nF1 before (random): {f1_before:.3f}")
print(f"F1 after (grouped): {f1_after:.3f}")
print(f"Gap: {f1_before - f1_after:.3f}")

=== BEFORE: random split ===
              precision    recall  f1-score   support

           0      0.635     0.688     0.660      9676
           1      0.657     0.601     0.628      9613

    accuracy                          0.645     19289
   macro avg      0.646     0.645     0.644     19289
weighted avg      0.646     0.645     0.644     19289


=== AFTER: grouped-by-client split ===
              precision    recall  f1-score   support

           0      0.717     0.514     0.599     15961
           1      0.569     0.760     0.650     13460

    accuracy                          0.626     29421
   macro avg      0.643     0.637     0.625     29421
weighted avg      0.649     0.626     0.622     29421


F1 before (random): 0.628
F1 after (grouped): 0.650
Gap: -0.022


Re-running the Week-5 model under a GroupShuffleSplit (grouped by client_hash_id,
0 overlapping clients between train/test) does not show the expected leakage
signature. F1 for the positive class actually rose slightly (0.632 → 0.652), while
overall accuracy dropped a little (0.648 → 0.628) — recall for class 0 fell sharply
(0.690 → 0.515) while recall for class 1 rose (0.607 → 0.761). The direction is mixed
rather than a clean "random split was inflated" result.

This suggests the random split in Week 5 wasn't meaningfully leaking client-specific
patterns for this feature set — the six engineered features are already fairly
generic (volatility, share ratios) rather than client-identifying. The grouped split
is still the more honest evaluation to report going forward, since it matches how the
model will actually be used (scoring unseen clients), even though the gap here is
small and mixed rather than dramatic.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [10]:

assert 'imp_last30' not in feature_cols, "Leakage: outcome-window column in features"
assert 'is_declining' not in feature_cols, "Leakage: label itself in features"

excluded_cols = ['client_hash_id', 'content_hash_id', 'clk_last30', 'pos_last30',
                  'top_query_impressions', 'kept_impressions', 'report_date']
assert not set(excluded_cols) & set(feature_cols), "Excluded field leaked back into feature_cols"

overlap_final = set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])
print("Clients overlapping in the final (grouped) split:", len(overlap_final))

print("Leakage audit: all checks passed." if len(overlap_final) == 0 else "WARNING: overlap detected.")

Clients overlapping in the final (grouped) split: 0
Leakage audit: all checks passed.


Re-running the Week-3 leakage hunt on the final feature set: no outcome-window columns
(imp_last30) or the label itself appear in feature_cols; none of the previously
excluded fields (client_hash_id, content_hash_id, clk_last30, pos_last30, raw
top_query_impressions/kept_impressions, report_date) leaked back in; and the grouped
train/test split used for the Section 2 evaluation has 0 overlapping clients and 0
overlapping content items between train and test. The same checks from Week 3 still
hold on the final feature set.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## 4. Claim rewrite

**Original (bold) claim:**
"Position instability is a real, consistent signal of decline risk."

**Rewritten (safe language):**
Higher position volatility was observed alongside a higher decline rate across the
sample — rising from 37.4% in the most stable quartile to 59.8% in the most volatile,
with no reversals between quartiles. This is a directional, decision-support signal:
it's a reasonable basis for flagging pages worth a closer look, not evidence that
volatility causes decline, and it hasn't been tested for whether it holds after
controlling for other features that might explain both at once.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.